# Importations

In [1]:
# Numerical and scientific python programming
import numpy as np

from scipy.stats import special_ortho_group

# Auxiliary python functions
from itertools import combinations, product

# Local importations
from moments.bloch import (compute_pauli_basis, compute_tensor_basis, compute_subset_index_map,
                           compute_bloch_vector, compute_dm_from_bloch, compute_bloch_norms_from_vector)
from moments.quantum import generate_rand_dm, compute_is_valid_dm, compute_concurrence, compute_negativity

# Definitions

In [2]:
def compute_rand_SO_subset(subset_index_map):
    Q = {}
    for subset in subset_index_map.keys():
        Q[subset] = special_ortho_group.rvs(len(subset_index_map[subset]))
    return Q

def check_tensor_rot(Q):
    for subset in Q.keys():
        if len(subset) > 1:
            Q_tensor = np.array([1])
            for m in subset:
                Q_tensor = np.kron(Q_tensor, Q[(m,)])
            if np.allclose(Q[subset], Q_tensor):
                return False
    return True

In [3]:
dn, N = 2, 2
dim = [dn]*N
d = int(np.prod(dim))

pauli_basis = compute_pauli_basis()

local_bases = [pauli_basis.copy()] * N
local_basis_sizes = [len(basis) for basis in local_bases]

tensor_basis = compute_tensor_basis(local_bases)
subset_index_map = compute_subset_index_map(local_basis_sizes)

# Viability test

In [4]:
is_valid = False
j = 0

while not is_valid:
    j += 1
    
    rho = generate_rand_dm(d, d)
    r = compute_bloch_vector(tensor_basis, subset_index_map, rho)

    Q = compute_rand_SO_subset(subset_index_map)
    if not check_tensor_rot(Q):
        print("Warning: Q_M is equal to the tensor product of Q_m for some M")
    
    r_rot = {subset: Q[subset] @ r[subset] for subset in subset_index_map.keys()}
    
    rho_rot = compute_dm_from_bloch(tensor_basis, subset_index_map, r_rot)
    is_valid, _ = compute_is_valid_dm(rho_rot)

print("Sucess:", is_valid)
print("Number of iterations:", j)


Sucess: True
Number of iterations: 2


# Local identity rotations

$$
Q_1 = Q_2 = \mathbb I_3
$$

In [ ]:
is_valid = False
j = 0

while not is_valid:
    j += 1

    rho = generate_rand_dm(d, d)
    r = compute_bloch_vector(tensor_basis, subset_index_map, rho)

    Q_12 = special_ortho_group.rvs(9)
    if np.allclose(Q_12, np.identity(9)):
        print("Warning: Q_12 is equal to the tensor product of Q_1 and Q_2 (i.e. the identity).")
    
    r_rot = r.copy()
    r_rot[(1, 2)] = Q_12 @ r_rot[(1, 2)]
    
    rho_rot = compute_dm_from_bloch(tensor_basis, subset_index_map, r_rot)
    
    is_valid, _ = compute_is_valid_dm(rho_rot)

print("Sucess:", is_valid)
print("Number of iterations:", j)

Sucess: True
Number of iterations: 17


In [6]:
R = compute_bloch_norms_from_vector(r)
R_rot = compute_bloch_norms_from_vector(r_rot)

bloch_diff = {subset: np.allclose(r[subset], r_rot[subset]) for subset in subset_index_map}
length_diff = {subset: np.allclose(R[subset], R_rot[subset]) for subset in subset_index_map}
DC = abs(compute_concurrence(rho) - compute_concurrence(rho_rot))
DN = abs(compute_negativity(rho = rho) - compute_negativity(rho = rho_rot))

print("One-body Bloch vectors are constant:", (bloch_diff[(1,)] and bloch_diff[(2,)]))
print("Two-body Bloch vector is constant:", bloch_diff[(1, 2)])
print("Density matrix is constant:", np.allclose(rho, rho_rot))
print("Bloch lengths are constant:", all(list(length_diff.values())))
print(f"Concurrences difference: {DC:.4f}")
print(f"Negativity difference: {DN:.4f}")

One-body Bloch vectors are constant: True
Two-body Bloch vector is constant: False
Density matrix is constant: False
Bloch lengths are constant: True
Concurrences difference: 0.0888
Negativity difference: 0.0336


# Exact rotations

In [11]:
def iterate_k_nonzero_arrays(k, size = 9, min_value = 1, max_value = 10):
    
    base_array = np.zeros(size)
    
    value_range = range(min_value, max_value + 1)
    
    for indices in combinations(range(size), k):
        
        for values in product(value_range, repeat = k):
            
            array = base_array.copy()
            array[list(indices)] = values
            
            yield array

In [ ]:
a = 0.5
r = {(1,): a * np.array([0, 0, 1]), (2,): a * np.array([0, 0, 1]), (1, 2): np.array([0, 0, 0, 0, 0, 0, 0, 0, 1])}

rho = compute_dm_from_bloch(tensor_basis, subset_index_map, r)
is_valid, _ = compute_is_valid_dm(rho)

C, N = compute_concurrence(rho = rho), compute_negativity(rho=rho)

print("Original state:\n")
print(rho)
print("\nIs a valid density matrix?", is_valid)
print("\nConcurrence:", C)
print("Negativity:", N)

Original state:

[[0.75+0.j 0.  +0.j 0.  +0.j 0.  +0.j]
 [0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j]
 [0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j]
 [0.  +0.j 0.  +0.j 0.  +0.j 0.25+0.j]]

Is a valid density matrix? True

Concurrence: 0.0
Negativity: 0.0


In [14]:
k = 4
results, concurrences, negativities = [], [], []

for j, r_12 in enumerate(iterate_k_nonzero_arrays(k, max_value = 3)):
    
    r_rot = r.copy()
    r_rot[(1, 2)] = r_12 / np.linalg.norm(r_12)
    rho_rot = compute_dm_from_bloch(tensor_basis, subset_index_map, r_rot)
    
    is_valid, _ = compute_is_valid_dm(rho_rot)
    if is_valid:
        print(f'\n# {j}:\nr_12 = {r_12}.')
        C_rot, N_rot = compute_concurrence(rho=rho_rot), compute_negativity(rho=rho_rot)
        print(f"C = {C_rot:.4f}")
        print(f"N = {N_rot:.4f}")
        results.append(j)
        concurrences.append(C_rot)
        negativities.append(N_rot)


# 810:
r_12 = [1. 1. 0. 1. 0. 0. 0. 0. 1.].
C = 0.3090
N = 0.1545

# 823:
r_12 = [1. 2. 0. 2. 0. 0. 0. 0. 2.].
C = 0.3491
N = 0.1746

# 824:
r_12 = [1. 2. 0. 2. 0. 0. 0. 0. 3.].
C = 0.3395
N = 0.1697

# 827:
r_12 = [1. 2. 0. 3. 0. 0. 0. 0. 3.].
C = 0.3444
N = 0.1722

# 833:
r_12 = [1. 3. 0. 2. 0. 0. 0. 0. 3.].
C = 0.3444
N = 0.1722

# 836:
r_12 = [1. 3. 0. 3. 0. 0. 0. 0. 3.].
C = 0.3582
N = 0.1791

# 850:
r_12 = [2. 2. 0. 2. 0. 0. 0. 0. 2.].
C = 0.3090
N = 0.1545

# 863:
r_12 = [2. 3. 0. 3. 0. 0. 0. 0. 3.].
C = 0.3374
N = 0.1687

# 890:
r_12 = [3. 3. 0. 3. 0. 0. 0. 0. 3.].
C = 0.3090
N = 0.1545

# 4374:
r_12 = [1. 0. 0. 0. 0. 1. 0. 1. 1.].
C = 0.5000
N = 0.1830

# 4414:
r_12 = [2. 0. 0. 0. 0. 2. 0. 2. 2.].
C = 0.5000
N = 0.1830

# 4454:
r_12 = [3. 0. 0. 0. 0. 3. 0. 3. 3.].
C = 0.5000
N = 0.1830

# 4862:
r_12 = [0. 1. 1. 1. 0. 0. 0. 0. 3.].
C = 0.2532
N = 0.1217

# 4892:
r_12 = [0. 2. 1. 2. 0. 0. 0. 0. 3.].
C = 0.3463
N = 0.1696

# 4919:
r_12 = [0. 3. 1. 2. 0. 0. 0. 0. 3.].
C = 0.3560


In [ ]:
j_max = np.argmax(np.array(concurrences))
k_max = np.argmax(np.array(negativities))

print("Maximum concurrence")
print(f"# {results[j_max]}:")
print(f"C = {concurrences[j_max]:.4f}")
print(f"N = {negativities[k_max]:.4f}")

if j_max != k_max:
    print("\nMaximums do not coincide!")

Maximum concurrence
# 8505:
C = 0.5000
N = 0.1830
